# FIFA World Cup 2026 — Player Match physical Collection

## 1. Project Setup

This notebook collects real FIFA World Cup 2026 player-level
statistics using the same public data source used for the
MatchStatistics dataset.

The source provides World Cup 2026 match details, goals, cards,
official FIFA match physical, top scorers, and player physical
data.

The final dataset will be prepared for the SQL player-statistics
table.

No fake, random, or generated player statistics will be used.

Before defining the final columns, we will inspect the available
player-level data from the source.

In [1]:
import requests
import pandas as pd
from pathlib import Path

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Test the Player Statistics Data Source

The player-level data will be collected from the same public
World Cup 2026 API used in the MatchStatistics notebook.

We will first test the API's `all` endpoint.

This step only verifies that the source is accessible and that
it returns the expected World Cup 2026 match data.

No data will be modified or saved yet.

In [2]:
api_url = "https://foot-boll.com/api/data.php?action=all"

response = requests.get(
    api_url,
    timeout=30,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Response size:", len(response.content), "bytes")

if response.status_code == 200:
    api_data = response.json()

    print("\nAPI request successful!")
    print("Response type:", type(api_data))

    if isinstance(api_data, list):
        print("Number of records:", len(api_data))
    elif isinstance(api_data, dict):
        print("Top-level keys:", list(api_data.keys()))
else:
    print("\nAPI request failed!")

Status code: 200
Content-Type: application/json; charset=utf-8
Response size: 77812 bytes

API request successful!
Response type: <class 'dict'>
Top-level keys: ['ok', 'action', 'updated', 'matches']


## 3. Inspect the Match Data Structure

The API successfully returned the World Cup 2026 match dataset.

We will inspect the structure of one match record to identify
which player-level information is available.

This will help us determine which real player statistics can be
collected for the PlayerMatchStats table.

No assumptions will be made about the available fields.

In [3]:
matches_data = api_data["matches"]

print("Matches type:", type(matches_data))
print("Number of matches:", len(matches_data))

if len(matches_data) > 0:
    first_match = matches_data[0]

    print("\nFirst match record:")
    print("=" * 80)

    if isinstance(first_match, dict):
        print("Keys:")
        for key in first_match.keys():
            print("-", key)

        print("\nFull first match record:")
        print(first_match)
    else:
        print("Unexpected match record type:", type(first_match))

Matches type: <class 'list'>
Number of matches: 104

First match record:
Keys:
- id
- round
- group
- team1
- team2
- team1_ar
- team2_ar
- flag1
- flag2
- status
- score
- penalties
- winner
- live_minute
- date
- time
- datetime
- ground

Full first match record:
{'id': 0, 'round': 'Matchday 1', 'group': 'Group A', 'team1': 'Mexico', 'team2': 'South Africa', 'team1_ar': 'المكسيك', 'team2_ar': 'جنوب أفريقيا', 'flag1': 'https://foot-boll.com/flag.php?s=w80&c=mx', 'flag2': 'https://foot-boll.com/flag.php?s=w80&c=za', 'status': 'finished', 'score': [2, 0], 'penalties': None, 'winner': 'team1', 'live_minute': None, 'date': '2026-06-11', 'time': '11:00 م', 'datetime': 1781204400, 'ground': 'Mexico City'}


## 4. Test the Detailed Match Endpoint

The `all` endpoint provides the list of World Cup 2026 matches.

The API also provides a `match` endpoint that returns detailed
information for an individual match.

We will request the first match and inspect its returned fields
to determine whether it contains player-level information such as
goals, cards, or player statistics.

No data will be modified or saved yet.

In [4]:
test_match_id = matches_data[0]["id"]

match_url = (
    "https://foot-boll.com/api/data.php"
    f"?action=match&id={test_match_id}"
)

match_response = requests.get(
    match_url,
    timeout=30,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("Test Match ID:", test_match_id)
print("Status code:", match_response.status_code)
print("Content-Type:", match_response.headers.get("Content-Type"))
print("Response size:", len(match_response.content), "bytes")

if match_response.status_code == 200:
    match_data = match_response.json()

    print("\nDetailed match request successful!")
    print("Response type:", type(match_data))

    if isinstance(match_data, dict):
        print("\nTop-level keys:")
        for key in match_data.keys():
            print("-", key)
else:
    print("\nDetailed match request failed!")

Test Match ID: 0
Status code: 200
Content-Type: application/json; charset=utf-8
Response size: 4487 bytes

Detailed match request successful!
Response type: <class 'dict'>

Top-level keys:
- ok
- action
- updated
- match


## 5. Inspect the Detailed Match Structure

The detailed match endpoint successfully returned a `match` object.

We will inspect its structure to identify all available fields
and determine where player-level information is stored.

We will specifically look for:

- Goals and goal scorers
- Player names or IDs
- Cards
- Player statistics
- Match events
- Physical statistics

No data will be modified at this stage.

In [5]:
detailed_match = match_data["match"]

print("Detailed match type:", type(detailed_match))

if isinstance(detailed_match, dict):

    print("\nDetailed match keys:")
    print("=" * 80)

    for key in detailed_match.keys():
        value = detailed_match[key]

        print(
            f"- {key} "
            f"(type: {type(value).__name__})"
        )

else:
    print("Unexpected match structure:", type(detailed_match))

Detailed match type: <class 'dict'>

Detailed match keys:
- id (type: int)
- round (type: str)
- group (type: str)
- team1 (type: str)
- team2 (type: str)
- team1_ar (type: str)
- team2_ar (type: str)
- flag1 (type: str)
- flag2 (type: str)
- status (type: str)
- score (type: list)
- penalties (type: NoneType)
- winner (type: str)
- live_minute (type: NoneType)
- date (type: str)
- time (type: str)
- datetime (type: int)
- ground (type: str)
- ht (type: list)
- goals1 (type: list)
- goals2 (type: list)
- cards (type: list)
- stats (type: list)


## 6. Inspect Player Events

The detailed match response contains separate fields for:

- `goals1`
- `goals2`
- `cards`

We will inspect these fields from the first match to determine
whether they contain player names, event minutes, event types, or
other player-level information.

No data will be modified at this stage.

In [6]:
print("GOALS — TEAM 1")
print("=" * 80)
print(detailed_match["goals1"])

print("\nGOALS — TEAM 2")
print("=" * 80)
print(detailed_match["goals2"])

print("\nCARDS")
print("=" * 80)
print(detailed_match["cards"])

GOALS — TEAM 1
[{'name': 'Julián Quiñones', 'minute': '9'}, {'name': 'Raúl Jiménez', 'minute': '67'}]

GOALS — TEAM 2
[]

CARDS
[{'team': 2, 'minute': 17, 'name': 'Teboho Mokoena', 'type': 'yellow', 'reason_ar': 'خطأ قويّ', 'reason_en': 'a bad foul'}, {'team': 1, 'minute': 23, 'name': 'Brian Gutiérrez', 'type': 'yellow', 'reason_ar': 'خطأ قويّ', 'reason_en': 'a bad foul'}, {'team': 2, 'minute': 49, 'name': 'Sphephelo Sithole', 'type': 'red'}, {'team': 2, 'minute': 74, 'name': 'Nkosinathi Sibisi', 'type': 'yellow', 'reason_ar': 'خطأ قويّ', 'reason_en': 'a bad foul'}, {'team': 2, 'minute': 84, 'name': 'Themba Zwane', 'type': 'red', 'reason_ar': 'سلوك عنيف', 'reason_en': 'violent conduct'}, {'team': 1, 'minute': 90, 'name': 'César Montes', 'type': 'red'}]


## 7. Inspect Match Statistics

The detailed match response contains a `stats` field.

We will inspect its structure to determine whether it contains
player-level statistics or only team-level match statistics.

We will identify:

- Statistic names
- Team values
- Player information, if available
- The structure of each statistic

No data will be modified at this stage.

In [7]:
match_stats = detailed_match["stats"]

print("Stats type:", type(match_stats))
print("Number of stat records:", len(match_stats))

print("\nStatistics:")
print("=" * 80)

for i, stat in enumerate(match_stats, start=1):
    print(f"\nSTAT {i}")
    print("-" * 80)

    if isinstance(stat, dict):
        print("Keys:", list(stat.keys()))
        print("Data:", stat)
    else:
        print("Type:", type(stat))
        print("Data:", stat)

Stats type: <class 'list'>
Number of stat records: 9

Statistics:

STAT 1
--------------------------------------------------------------------------------
Keys: ['k', 'k_en', 'v', 'unit']
Data: {'k': 'الاستحواذ', 'k_en': 'Possession', 'v': [61, 40], 'unit': '%'}

STAT 2
--------------------------------------------------------------------------------
Keys: ['k', 'k_en', 'v', 'unit']
Data: {'k': 'إجمالي التسديدات', 'k_en': 'Shots', 'v': [16, 3], 'unit': ''}

STAT 3
--------------------------------------------------------------------------------
Keys: ['k', 'k_en', 'v', 'unit']
Data: {'k': 'تسديدات على المرمى', 'k_en': 'Shots on target', 'v': [4, 2], 'unit': ''}

STAT 4
--------------------------------------------------------------------------------
Keys: ['k', 'k_en', 'v', 'unit']
Data: {'k': 'ركلات ركنية', 'k_en': 'Corners', 'v': [3, 1], 'unit': ''}

STAT 5
--------------------------------------------------------------------------------
Keys: ['k', 'k_en', 'v', 'unit']
Data: {'k': 'تسلّ

## 8. Inspect the Available Player Data Endpoints

The detailed match endpoint provides player-level events such as
goals and cards, but its `stats` field contains only team-level
statistics.

The API documentation indicates that additional player-level data
may be available through separate endpoints.

We will request the public API documentation and inspect the
documented actions/endpoints related to players, scorers, and
physical statistics.

No endpoint will be guessed.

In [8]:
docs_url = "https://foot-boll.com/api.php?lang=en"

docs_response = requests.get(
    docs_url,
    timeout=30,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("Status code:", docs_response.status_code)
print("Content-Type:", docs_response.headers.get("Content-Type"))
print("Response size:", len(docs_response.content), "bytes")

if docs_response.status_code == 200:
    docs_text = docs_response.text

    print("\nAPI documentation loaded successfully!")

    keywords = [
        "player",
        "scorer",
        "physical",
        "statistics",
        "endpoint"
    ]

    for keyword in keywords:
        print(
            f"'{keyword}':",
            docs_text.lower().count(keyword)
        )
else:
    print("\nAPI documentation request failed!")

Status code: 200
Content-Type: text/html; charset=UTF-8
Response size: 37844 bytes

API documentation loaded successfully!
'player': 3
'scorer': 19
'physical': 4
'statistics': 2
'endpoint': 1


## 9. Identify the Documented Player Data Actions

The API documentation contains references to player, scorer, and
physical data.

We will search the documentation for the surrounding text of
these terms to identify the exact API actions and endpoints that
the source provides.

We will not guess any endpoint.

In [9]:
keywords = [
    "player",
    "scorer",
    "physical"
]

for keyword in keywords:

    print("\n" + "=" * 100)
    print("KEYWORD:", keyword)
    print("=" * 100)

    text_lower = docs_text.lower()
    start = 0
    count = 0

    while True:
        position = text_lower.find(keyword, start)

        if position == -1:
            break

        count += 1

        print("\n--- Occurrence", count, "---")

        print(
            docs_text[
                max(0, position - 500):
                position + 1000
            ]
        )

        start = position + len(keyword)

        if count >= 10:
            print("\nShowing first 10 occurrences only.")
            break


KEYWORD: player

--- Occurrence 1 ---

            Analytics hub<i class="nav-caret">▾</i>

          </button>

          <div class="nav-drop">

            <a href="https://foot-boll.com/dashboard.php?lang=en">Data dashboard</a>

            <a href="https://foot-boll.com/compare.php?lang=en">Team compare</a>

            <a href="https://foot-boll.com/stats.php?lang=en">Statistics</a>

            <a href="https://foot-boll.com/physical.php?lang=en">Physical data</a>

            <a href="https://foot-boll.com/motm.php?lang=en">Player of the match</a>

            <a href="https://foot-boll.com/topscorers.php?lang=en">Top Scorers</a>

            <a href="https://foot-boll.com/bookings.php?lang=en">Cards</a>

            <a href="https://foot-boll.com/referees.php?lang=en">Referees</a>

          </div>

        </div>



        <div class="nav-group">

          <button type="button" class="nav-group-btn">

            foot-boll reads<i class="nav-caret">▾</i>

          </butto

## 10. Extract the Player Data API Documentation

The API documentation confirms that player-related data exists,
including scorer and physical data.

The previous output was truncated, so we will now extract the
relevant documentation sections directly.

We will search for:

- `scorers`
- `physical`
- API action names
- Example request URLs

This will allow us to identify the exact documented endpoints
before making any further requests.

In [10]:
for keyword in ["scorers", "physical"]:

    print("\n" + "=" * 100)
    print("DOCUMENTATION FOR:", keyword)
    print("=" * 100)

    text_lower = docs_text.lower()
    positions = []
    start = 0

    while True:
        position = text_lower.find(keyword, start)

        if position == -1:
            break

        positions.append(position)
        start = position + len(keyword)

    print("Occurrences found:", len(positions))

    for i, position in enumerate(positions, start=1):
        print("\n" + "-" * 80)
        print("Occurrence", i)
        print("-" * 80)

        print(
            docs_text[
                max(0, position - 1200):
                position + 2500
            ]
        )


DOCUMENTATION FOR: scorers
Occurrences found: 18

--------------------------------------------------------------------------------
Occurrence 1
--------------------------------------------------------------------------------

<!DOCTYPE html>

<html lang="en" dir="ltr">

<head>

<meta charset="UTF-8">

<meta name="viewport" content="width=device-width, initial-scale=1">

<script>try{if(!localStorage.getItem('wc_seen')){document.documentElement.classList.add('intro');localStorage.setItem('wc_seen','1');}}catch(e){}</script>

<title>Developer API — World Cup 2026 | foot-boll Sports Hub</title>

<meta name="description" content="Free, public JSON API for the FIFA World Cup 2026: matches, live scores, goals &amp; cards, official FIFA stats, top scorers and group standings. No key, no signup.">



<meta name="keywords" content="World Cup 2026, live scores, predictions, FIFA stats, foot-boll">



<meta name="theme-color" content="#dfeaf4">


<link rel="manifest" href="https://foot-boll.com/m

## 11. Extract Exact Player API Endpoints

The API documentation confirms the existence of player-related
data, but the previous output was truncated.

We will extract only lines containing API actions, endpoint paths,
and request examples.

This will allow us to identify the exact endpoints for:

- Top scorers
- Physical player data
- Other player-level statistics

No endpoint will be guessed.

In [11]:
import re

# Remove HTML tags so the documentation is easier to inspect
clean_docs = re.sub(r"<[^>]+>", " ", docs_text)

# Decode common HTML entities
clean_docs = (
    clean_docs
    .replace("&amp;", "&")
    .replace("&lt;", "<")
    .replace("&gt;", ">")
    .replace("&quot;", '"')
)

# Collapse excessive whitespace
clean_docs = re.sub(r"\s+", " ", clean_docs)

# Find API-related URL/action patterns
patterns = [
    r"action=[A-Za-z0-9_]+",
    r"api/data\.php[^ <\"']*",
    r"data\.[A-Za-z_]+",
    r"scorers",
    r"physical"
]

for pattern in patterns:

    print("\n" + "=" * 100)
    print("PATTERN:", pattern)
    print("=" * 100)

    matches = sorted(set(re.findall(pattern, clean_docs, re.IGNORECASE)))

    for match in matches:
        print(match)


PATTERN: action=[A-Za-z0-9_]+
action=match
action=results
action=scorers
action=token

PATTERN: api/data\.php[^ <\"']*
api/data.php
api/data.php?action=
api/data.php?action=match&id=12
api/data.php?action=results
api/data.php?action=scorers

PATTERN: data\.[A-Za-z_]+
data.php
data.scorers

PATTERN: scorers
Scorers
scorers

PATTERN: physical
Physical
physical


## 12. Test the Top Scorers Endpoint

The API documentation explicitly provides the `scorers` action:

`api/data.php?action=scorers`

This endpoint is documented as providing top-scorer data.

We will request it and inspect its response structure to determine
whether it contains player names, teams, goals, assists, or other
player-level information.

No data will be modified or saved yet.

In [12]:
scorers_url = (
    "https://foot-boll.com/api/data.php"
    "?action=scorers"
)

scorers_response = requests.get(
    scorers_url,
    timeout=30,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("Status code:", scorers_response.status_code)
print("Content-Type:", scorers_response.headers.get("Content-Type"))
print("Response size:", len(scorers_response.content), "bytes")

if scorers_response.status_code == 200:
    scorers_data = scorers_response.json()

    print("\nScorers request successful!")
    print("Response type:", type(scorers_data))

    if isinstance(scorers_data, dict):
        print("\nTop-level keys:")
        for key in scorers_data.keys():
            print("-", key)

    elif isinstance(scorers_data, list):
        print("Number of records:", len(scorers_data))
else:
    print("\nScorers request failed!")

Status code: 200
Content-Type: application/json; charset=utf-8
Response size: 20204 bytes

Scorers request successful!
Response type: <class 'dict'>

Top-level keys:
- ok
- action
- updated
- scorers


## 13. Inspect the Scorer Data Structure

The `scorers` endpoint successfully returned player-level data.

We will inspect:

- The type of the `scorers` object
- The number of player records
- The fields available for each player
- A small sample of the actual records

This will tell us exactly which real player statistics are
available from this endpoint.

No data will be modified or saved yet.

In [13]:
scorers = scorers_data["scorers"]

print("Scorers type:", type(scorers))
print("=" * 80)

if isinstance(scorers, list):

    print("Number of scorer records:", len(scorers))

    if len(scorers) > 0:
        print("\nFirst scorer record:")
        print(scorers[0])

        print("\nFields:")
        for key in scorers[0].keys():
            print("-", key)

elif isinstance(scorers, dict):

    print("Number of scorer groups/keys:", len(scorers))
    print("\nKeys:")
    for key in scorers.keys():
        print("-", key)

    print("\nSample content:")
    print(scorers)

else:
    print("Unexpected scorers structure:", type(scorers))

Scorers type: <class 'list'>
Number of scorer records: 178

First scorer record:
{'name': 'Kylian Mbappé', 'team': 'France', 'goals': 10}

Fields:
- name
- team
- goals


## 14. Identify the Player Physical Data Endpoint

The API documentation contains a separate Physical Data section.

The `scorers` endpoint provides tournament-level player scoring
data, but our PlayerMatchStats table requires match-level data.

We will inspect the documentation specifically around the
Physical Data section to identify its exact API endpoint and
available fields.

No endpoint will be guessed.

In [14]:
text_lower = clean_docs.lower()

keyword = "physical"

positions = []
start = 0

while True:
    position = text_lower.find(keyword, start)

    if position == -1:
        break

    positions.append(position)
    start = position + len(keyword)

print("Physical occurrences found:", len(positions))

for i, position in enumerate(positions, start=1):

    print("\n" + "=" * 100)
    print("OCCURRENCE", i)
    print("=" * 100)

    print(
        clean_docs[
            max(0, position - 1500):
            position + 3000
        ]
    )

Physical occurrences found: 3

OCCURRENCE 1
 try{if(!localStorage.getItem('wc_seen')){document.documentElement.classList.add('intro');localStorage.setItem('wc_seen','1');}}catch(e){} Developer API — World Cup 2026 | foot-boll Sports Hub {"@context":"https:\/\/schema.org","@graph":[{"@type":"WebSite","@id":"https:\/\/foot-boll.com\/#website","url":"https:\/\/foot-boll.com\/","name":"foot-boll","description":"Free, public JSON API for the FIFA World Cup 2026: matches, live scores, goals \u0026 cards, official FIFA stats, top scorers and group standings. No key, no signup.","inLanguage":"en"},{"@type":"Organization","@id":"https:\/\/foot-boll.com\/#org","name":"foot-boll","url":"https:\/\/foot-boll.com\/","logo":"https:\/\/foot-boll.com\/assets\/img\/og.png"}]} Arabic analytics hub for World Cup 2026 📱 Get app العربية Home Fixtures ▾ Today&#039;s schedule All Matches Knockout Analytics hub ▾ Data dashboard Team compare Statistics Physical data Player of the match Top Scorers Cards Referee

## 15. Test the Player Physical Data Endpoint

The API documentation explicitly lists:

`physical — Per-player running data from FIFA reports`

We will request the documented `physical` action and inspect
the returned structure.

This is important because these are player-level data from FIFA
reports and may provide the real player statistics needed for
our player dataset.

No data will be modified or saved yet.

In [15]:
physical_url = (
    "https://foot-boll.com/api/data.php"
    "?action=physical"
)

physical_response = requests.get(
    physical_url,
    timeout=30,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("Status code:", physical_response.status_code)
print("Content-Type:", physical_response.headers.get("Content-Type"))
print("Response size:", len(physical_response.content), "bytes")

if physical_response.status_code == 200:
    physical_data = physical_response.json()

    print("\nPhysical data request successful!")
    print("Response type:", type(physical_data))

    if isinstance(physical_data, dict):
        print("\nTop-level keys:")
        for key in physical_data.keys():
            print("-", key)

    elif isinstance(physical_data, list):
        print("Number of records:", len(physical_data))
else:
    print("\nPhysical data request failed!")

Status code: 200
Content-Type: application/json; charset=utf-8
Response size: 1063162 bytes

Physical data request successful!
Response type: <class 'dict'>

Top-level keys:
- ok
- action
- updated
- players


## 16. Inspect the Player Physical Data Structure

The `physical` endpoint returned a large player-level dataset.

We will inspect:

- The type of `players`
- Number of player records
- Available fields for each player
- A sample player record

This will show us exactly which real player statistics are
available before we design the final PlayerMatchStats dataset.

No data will be modified or saved yet.

In [16]:
physical_players = physical_data["players"]

print("Players type:", type(physical_players))
print("=" * 80)

if isinstance(physical_players, list):

    print("Number of player records:", len(physical_players))

    if len(physical_players) > 0:

        print("\nFirst player record:")
        print(physical_players[0])

        print("\nFields:")
        for key in physical_players[0].keys():
            print("-", key)

elif isinstance(physical_players, dict):

    print("Number of player groups/keys:", len(physical_players))

    print("\nKeys:")
    for key in physical_players.keys():
        print("-", key)

else:
    print(
        "Unexpected players structure:",
        type(physical_players)
    )

Players type: <class 'list'>
Number of player records: 1029

First player record:
{'name': 'RODRI', 'team': 'Spain', 'teamAr': 'إسبانيا', 'flag': 'https://foot-boll.com/flag.php?s=w40&c=es', 'photo': 'https://digitalhub.fifa.com/transform/295ab5b2-4dbf-42b9-918b-04fb4ae0ec0a/RODRI_411375', 'num': 16, 'm': 8, 'dist': 96233.70000000001, 'sprints': 318, 'hsr': 1294, 'top': 31.5, 'zones': [30824.8, 44470.3, 15304.899999999998, 4504.400000000001, 1129.3], 'lb': [187, 171], 'lbv': [78, 57, 52, 182, 0, 5], 'cross': [1, 0, 0, 2, 0, 0, 3]}

Fields:
- name
- team
- teamAr
- flag
- photo
- num
- m
- dist
- sprints
- hsr
- top
- zones
- lb
- lbv
- cross


## 17. Inspect Physical Data Across Players

The physical endpoint provides 1,029 player records.

Some fields are directly understandable:

- `m` — matches
- `dist` — distance
- `sprints` — sprints
- `hsr` — high-speed running
- `top` — top speed

Other fields are arrays:

- `zones`
- `lb`
- `lbv`
- `cross`

Before designing the SQL table, we need to determine whether
these arrays have fixed meanings and consistent lengths across
all players.

We will inspect the lengths and values of these fields across
the complete dataset.

In [17]:
array_fields = [
    "zones",
    "lb",
    "lbv",
    "cross"
]

print("PHYSICAL DATA FIELD STRUCTURE")
print("=" * 80)

for field in array_fields:

    lengths = (
        physical_players
        and [
            len(player[field])
            for player in physical_players
            if isinstance(player.get(field), list)
        ]
    )

    print(f"\nField: {field}")
    print("Number of records with list:", len(lengths))

    if lengths:
        print("Unique lengths:", sorted(set(lengths)))
        print("Minimum length:", min(lengths))
        print("Maximum length:", max(lengths))

        print("First 5 examples:")

        shown = 0

        for player in physical_players:

            value = player.get(field)

            if isinstance(value, list):
                print(
                    f"  {player['name']} ({player['team']}):",
                    value
                )

                shown += 1

                if shown == 5:
                    break

PHYSICAL DATA FIELD STRUCTURE

Field: zones
Number of records with list: 1029
Unique lengths: [5]
Minimum length: 5
Maximum length: 5
First 5 examples:
  RODRI (Spain): [30824.8, 44470.3, 15304.899999999998, 4504.400000000001, 1129.3]
  Alexis MAC ALLISTER (Argentina): [32710, 41710.2, 11818.199999999999, 3470.1, 853.3000000000001]
  Marc CUCURELLA (Spain): [35446.1, 35667.5, 10387.8, 4750.799999999999, 1734]
  Pau CUBARSI (Spain): [37264.2, 34965.299999999996, 9070.3, 3338.2, 1317.9]
  Michael OLISE (France): [26863.100000000002, 37368, 12546.8, 5484.8, 2237]

Field: lb
Number of records with list: 1029
Unique lengths: [2]
Minimum length: 2
Maximum length: 2
First 5 examples:
  RODRI (Spain): [187, 171]
  Alexis MAC ALLISTER (Argentina): [76, 63]
  Marc CUCURELLA (Spain): [109, 87]
  Pau CUBARSI (Spain): [186, 178]
  Michael OLISE (France): [133, 103]

Field: lbv
Number of records with list: 1029
Unique lengths: [6]
Minimum length: 6
Maximum length: 6
First 5 examples:
  RODRI (Spain)

## 18. Identify the Meaning of Physical Data Arrays

The physical endpoint contains several array-based fields.

Before using these fields in the final dataset, we must identify
what each position represents.

We will search the API documentation for the exact descriptions
of:

- `zones`
- `lb`
- `lbv`
- `cross`

We will not assume the meaning of any array position.

In [18]:
array_keywords = ["zones", "lbv", "lb", "cross"]

print("SEARCHING API DOCUMENTATION")
print("=" * 80)

for keyword in array_keywords:

    print(f"\n{'=' * 80}")
    print(f"KEYWORD: {keyword}")
    print("=" * 80)

    text_lower = clean_docs.lower()
    start = 0
    found = 0

    while True:

        position = text_lower.find(keyword.lower(), start)

        if position == -1:
            break

        found += 1

        print(
            clean_docs[
                max(0, position - 1000):
                position + 2000
            ]
        )

        start = position + len(keyword)

        if found >= 5:
            break

    print(f"\nOccurrences shown: {found}")

SEARCHING API DOCUMENTATION

KEYWORD: zones

Occurrences shown: 0

KEYWORD: lbv

Occurrences shown: 0

KEYWORD: lb

Occurrences shown: 0

KEYWORD: cross

Occurrences shown: 0


## 18. Inspect the Physical Data Documentation

The API documentation does not explicitly mention the internal
field names `zones`, `lb`, `lbv`, or `cross`.

Therefore, we will inspect the documentation around the
`physical` endpoint itself and identify any descriptions,
examples, or field definitions provided by the source.

We will use only information supported by the source and will
not assign meanings to array positions without evidence.

In [19]:
id="step18_python"
text_lower = clean_docs.lower()

keyword = "physical"

positions = []
start = 0

while True:

    position = text_lower.find(keyword, start)

    if position == -1:
        break

    positions.append(position)
    start = position + len(keyword)

print("Physical occurrences found:", len(positions))

for i, position in enumerate(positions, start=1):

    print("\n" + "=" * 100)
    print(f"PHYSICAL OCCURRENCE {i}")
    print("=" * 100)

    print(
        clean_docs[
            max(0, position - 3000):
            position + 6000
        ]
    )

Physical occurrences found: 3

PHYSICAL OCCURRENCE 1
 try{if(!localStorage.getItem('wc_seen')){document.documentElement.classList.add('intro');localStorage.setItem('wc_seen','1');}}catch(e){} Developer API — World Cup 2026 | foot-boll Sports Hub {"@context":"https:\/\/schema.org","@graph":[{"@type":"WebSite","@id":"https:\/\/foot-boll.com\/#website","url":"https:\/\/foot-boll.com\/","name":"foot-boll","description":"Free, public JSON API for the FIFA World Cup 2026: matches, live scores, goals \u0026 cards, official FIFA stats, top scorers and group standings. No key, no signup.","inLanguage":"en"},{"@type":"Organization","@id":"https:\/\/foot-boll.com\/#org","name":"foot-boll","url":"https:\/\/foot-boll.com\/","logo":"https:\/\/foot-boll.com\/assets\/img\/og.png"}]} Arabic analytics hub for World Cup 2026 📱 Get app العربية Home Fixtures ▾ Today&#039;s schedule All Matches Knockout Analytics hub ▾ Data dashboard Team compare Statistics Physical data Player of the match Top Scorers Card

## 19. Inspect the Source's Physical Data Page

The API documentation confirms that the `physical` endpoint
contains per-player running data from FIFA reports.

However, the documentation does not define the individual
positions inside the array fields.

We will therefore inspect the source page that displays the
physical player data and look for labels or descriptions
associated with these fields.

We will not assign meanings unless the source provides evidence.

In [20]:
physical_page_url = "https://foot-boll.com/physical.php"

physical_page_response = requests.get(
    physical_page_url,
    timeout=30,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("Status code:", physical_page_response.status_code)
print("Content-Type:", physical_page_response.headers.get("Content-Type"))
print("Response size:", len(physical_page_response.content), "bytes")

if physical_page_response.status_code == 200:

    physical_page_html = physical_page_response.text

    print("\nPhysical data page loaded successfully!")

    print("\nSearching page for relevant terms:")
    print("=" * 80)

    page_lower = physical_page_html.lower()

    keywords = [
        "distance",
        "sprint",
        "speed",
        "zone",
        "cross",
        "physical",
        "running",
        "high-speed"
    ]

    for keyword in keywords:
        print(
            f"{keyword}:",
            page_lower.count(keyword.lower())
        )

else:
    print("\nPhysical data page request failed!")

Status code: 200
Content-Type: text/html; charset=UTF-8
Response size: 43738 bytes

Physical data page loaded successfully!

Searching page for relevant terms:
distance: 0
sprint: 18
speed: 0
zone: 9
cross: 9
physical: 9
running: 0
high-speed: 0


## 20. Inspect the Physical Data Page Structure

The physical data page contains references to sprint, zone, and
cross statistics.

We will inspect the HTML/JavaScript surrounding these terms to
determine how the API fields are mapped to the labels shown on
the website.

This will help us identify the meaning of the array fields
without making unsupported assumptions.

In [21]:
search_terms = [
    "sprint",
    "zone",
    "cross",
    "lbv",
    "zones",
    "lb"
]

page_lower = physical_page_html.lower()

print("PHYSICAL PAGE FIELD MAPPING SEARCH")
print("=" * 80)

for term in search_terms:

    print("\n" + "=" * 100)
    print(f"TERM: {term}")
    print("=" * 100)

    start = 0
    found = 0

    while True:

        position = page_lower.find(term.lower(), start)

        if position == -1:
            break

        found += 1

        print(
            physical_page_html[
                max(0, position - 1200):
                position + 2500
            ]
        )

        start = position + len(term)

        if found >= 5:
            break

    print(f"\nOccurrences shown: {found}")

PHYSICAL PAGE FIELD MAPPING SEARCH

TERM: sprint
ch" href="https://foot-boll.com/physical.php?lang=en">English</a>


        </div>

      </nav>

    </div>

  </div>

</header>


<main class="wrap fb-page">


<div class="page-head">
  <h1>🏃 البيانات البدنيّة</h1>
  <p class="muted">أرقام كل لاعب من تقارير FIFA الرسميّة — رتّب بأي عمود، ابحث، وبدّل بين الإجمالي والمعدّل لكل مباراة.</p>
</div>

<div class="phys-controls">
  <input type="search" id="physSearch" class="phys-input" placeholder="ابحث عن لاعب أو منتخب…">
  <select id="physTeam" class="phys-input phys-team">
    <option value="">كل المنتخبات</option>
  </select>
  <div class="phys-toggle">
    <button type="button" class="phys-mode is-on" data-mode="total">الإجمالي</button>
    <button type="button" class="phys-mode" data-mode="avg">لكل مباراة</button>
  </div>
</div>

<p id="physStatus" class="empty-note">جارٍ التحميل…</p>

<div class="lb-wrap">
  <table class="leaderboard phys-table" id="physTable" hidden>
    <thead>
    

## 20.1 Extract the Field Mapping Lines

The previous output was truncated because the HTML contains
large blocks of JavaScript.

Instead of printing large surrounding sections, we will extract
only the lines containing the physical API field names and the
HTML elements where they are displayed.

This will make the field mapping easier to verify.

In [22]:
import re

lines = physical_page_html.splitlines()

terms = [
    "val(r, 'dist')",
    "val(r, 'sprints')",
    "val(r, 'hsr')",
    "val(r, 'top')",
    "val(r, 'zones')",
    "val(r, 'lb')",
    "val(r, 'lbv')",
    "val(r, 'cross')"
]

print("RELEVANT PHYSICAL FIELD MAPPINGS")
print("=" * 100)

found_any = False

for line_number, line in enumerate(lines, start=1):

    if any(term.lower() in line.lower() for term in terms):

        found_any = True

        print(f"\nLINE {line_number}")
        print("-" * 100)
        print(line.strip())

if not found_any:
    print("No exact field mappings found with these patterns.")

print("\nDone.")

RELEVANT PHYSICAL FIELD MAPPINGS

LINE 460
----------------------------------------------------------------------------------------------------
MAX.dist = Math.max(MAX.dist, val(r, 'dist') / m);

LINE 461
----------------------------------------------------------------------------------------------------
MAX.sprints = Math.max(MAX.sprints, val(r, 'sprints') / m);

LINE 462
----------------------------------------------------------------------------------------------------
MAX.hsr = Math.max(MAX.hsr, val(r, 'hsr') / m);

LINE 463
----------------------------------------------------------------------------------------------------
MAX.top = Math.max(MAX.top, val(r, 'top'));

LINE 476
----------------------------------------------------------------------------------------------------
r.querySelector('[data-c=dist]').textContent    = (val(r, 'dist') / 1000 / f).toFixed(1);

LINE 477
----------------------------------------------------------------------------------------------------
r.queryS

## 20.2 Extract the Array Field Mappings

The physical page confirms the scalar fields `dist`, `sprints`,
`hsr`, and `top`.

The remaining fields are arrays:

- `zones`
- `lb`
- `lbv`
- `cross`

We will extract only the JavaScript statements that reference
these four fields so their meanings can be determined from the
source.

In [23]:
array_terms = [
    "zones",
    "lbv",
    "lb",
    "cross"
]

print("ARRAY FIELD MAPPINGS")
print("=" * 100)

for line_number, line in enumerate(lines, start=1):

    if any(
        re.search(
            rf"\b{re.escape(term)}\b",
            line,
            re.IGNORECASE
        )
        for term in array_terms
    ):

        # Only show lines that appear to use the API record `r`
        # or directly reference one of the physical fields.
        if (
            "val(r" in line
            or "zones" in line.lower()
            or "lbv" in line.lower()
            or "cross" in line.lower()
        ):
            print(f"LINE {line_number}: {line.strip()}")

print("\nDone.")

ARRAY FIELD MAPPINGS
LINE 375: .ph-zones{display:flex;height:13px;border-radius:7px;overflow:hidden}
LINE 376: .ph-zones i{display:block;height:100%}
LINE 516: function bar(zones){
LINE 518: for (i = 0; i < 5; i++) tot += (+zones[i] || 0);
LINE 521: for (i = 0; i < 5; i++) seg += '<i style="width:' + ((+zones[i]||0)/tot*100).toFixed(1) + '%;background:' + sh[i] + '"></i>';
LINE 522: return '<div class="ph-block"><div class="ph-blk-t">' + LBL.zonesT + '</div><div class="ph-zones">' + seg + '</div></div>';
LINE 524: function lbBlock(lb, lbv){
LINE 527: var line = LBL.dir[0] + ' ' + (+lbv[0]||0) + ' · ' + LBL.dir[1] + ' ' + (+lbv[1]||0) + ' · ' + LBL.dir[2] + ' ' + (+lbv[2]||0)
LINE 528: + '  —  ' + LBL.typ[0] + ' ' + (+lbv[3]||0) + ' · ' + LBL.typ[1] + ' ' + (+lbv[4]||0) + ' · ' + LBL.typ[2] + ' ' + (+lbv[5]||0);
LINE 532: function crossBlock(cross){
LINE 533: var total = +cross[6] || 0; if (total <= 0) return '';
LINE 547: var rich = bar(p.zones || []) + lbBlock(p.lb || [0, 0], p.lbv ||

## 21. Identify the Labels of the Physical Statistics

The source code shows that the arrays are displayed using
label definitions such as:

- `LBL.zonesT`
- `LBL.dir`
- `LBL.typ`

The exact labels will tell us what each array position means.

We will extract these label definitions directly from the
physical data page.

In [24]:
label_terms = [
    "LBL.zonesT",
    "LBL.dir",
    "LBL.typ"
]

print("PHYSICAL STATISTICS LABEL DEFINITIONS")
print("=" * 100)

for line_number, line in enumerate(lines, start=1):

    if any(term.lower() in line.lower() for term in label_terms):
        print(f"LINE {line_number}:")
        print(line.strip())
        print("-" * 100)

print("\nSearching for cross-related labels...")

for line_number, line in enumerate(lines, start=1):

    if "cross" in line.lower() and "LBL" in line:

        print(f"LINE {line_number}:")
        print(line.strip())
        print("-" * 100)

print("\nDone.")

PHYSICAL STATISTICS LABEL DEFINITIONS
LINE 522:
return '<div class="ph-block"><div class="ph-blk-t">' + LBL.zonesT + '</div><div class="ph-zones">' + seg + '</div></div>';
----------------------------------------------------------------------------------------------------
LINE 527:
var line = LBL.dir[0] + ' ' + (+lbv[0]||0) + ' · ' + LBL.dir[1] + ' ' + (+lbv[1]||0) + ' · ' + LBL.dir[2] + ' ' + (+lbv[2]||0)
----------------------------------------------------------------------------------------------------
LINE 528:
+ '  —  ' + LBL.typ[0] + ' ' + (+lbv[3]||0) + ' · ' + LBL.typ[1] + ' ' + (+lbv[4]||0) + ' · ' + LBL.typ[2] + ' ' + (+lbv[5]||0);
----------------------------------------------------------------------------------------------------

Searching for cross-related labels...
LINE 534:
return '<div class="ph-block"><div class="ph-blk-t">' + LBL.crossesT + '</div><div class="ph-chips">' + chip(total, LBL.total) + '</div></div>';
-------------------------------------------------------

## 21.1 Locate the Physical Statistics Label Definitions

The page uses `LBL.zonesT`, `LBL.dir`, `LBL.typ`, `LBL.crossesT`,
and `LBL.total`, but their definitions are located elsewhere
in the page JavaScript.

We will locate the actual `LBL = ...` definition and inspect
the labels directly.`

In [25]:
print("SEARCHING FOR LBL DEFINITION")
print("=" * 100)

for line_number, line in enumerate(lines, start=1):

    if re.search(r"\bLBL\s*=", line):

        print(f"LINE {line_number}:")
        print(line.strip())
        print("-" * 100)

print("\nSearching for individual label properties...")

label_patterns = [
    r"zonesT\s*:",
    r"dir\s*:",
    r"typ\s*:",
    r"crossesT\s*:",
    r"total\s*:"
]

for pattern in label_patterns:

    print(f"\nPattern: {pattern}")

    found = 0

    for line_number, line in enumerate(lines, start=1):

        if re.search(pattern, line, re.IGNORECASE):

            print(f"LINE {line_number}: {line.strip()}")

            found += 1

            if found >= 10:
                break

    print("Occurrences shown:", found)

print("\nDone.")

SEARCHING FOR LBL DEFINITION
LINE 390:
var LBL = {
----------------------------------------------------------------------------------------------------

Searching for individual label properties...

Pattern: zonesT\s*:
LINE 399: zonesT:  "مناطق السرعة (1→5)",
Occurrences shown: 1

Pattern: dir\s*:
LINE 406: dir:     ["عبر","حول","فوق"],
Occurrences shown: 1

Pattern: typ\s*:
LINE 407: typ:     ["تمريرة","عرضيّة","تقدّم"],
Occurrences shown: 1

Pattern: crossesT\s*:
LINE 404: crossesT:"العرضيّات",
Occurrences shown: 1

Pattern: total\s*:
LINE 405: total:   "إجمالي",
Occurrences shown: 1

Done.


## 22. Extract the English Physical Statistics Labels

The physical page currently exposes Arabic labels in its
JavaScript definitions.

Because our project dataset and SQL schema are in English,
we will inspect the English version of the page and extract
the corresponding English labels.

This will allow us to define clear English column names without
guessing or translating the labels ourselves.

In [26]:
physical_page_en_url = "https://foot-boll.com/physical.php?lang=en"

physical_page_en_response = requests.get(
    physical_page_en_url,
    timeout=30,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("Status code:", physical_page_en_response.status_code)
print("Content-Type:", physical_page_en_response.headers.get("Content-Type"))
print("Response size:", len(physical_page_en_response.content), "bytes")

if physical_page_en_response.status_code == 200:

    physical_page_en_html = physical_page_en_response.text
    physical_en_lines = physical_page_en_html.splitlines()

    print("\nEnglish physical data page loaded successfully!")

    print("\nSearching for label definition:")
    print("=" * 100)

    for line_number, line in enumerate(physical_en_lines, start=1):

        if re.search(r"\bLBL\s*=", line):

            print(f"LINE {line_number}:")
            print(line.strip())
            print("-" * 100)

    print("\nSearching for relevant label properties:")
    print("=" * 100)

    label_patterns = [
        r"zonesT\s*:",
        r"dir\s*:",
        r"typ\s*:",
        r"crossesT\s*:",
        r"total\s*:"
    ]

    for pattern in label_patterns:

        for line_number, line in enumerate(
            physical_en_lines,
            start=1
        ):

            if re.search(pattern, line, re.IGNORECASE):

                print(
                    f"LINE {line_number}: "
                    f"{line.strip()}"
                )

else:
    print("\nEnglish physical data page request failed!")

Status code: 200
Content-Type: text/html; charset=UTF-8
Response size: 41354 bytes

English physical data page loaded successfully!

Searching for label definition:
LINE 390:
var LBL = {
----------------------------------------------------------------------------------------------------

Searching for relevant label properties:
LINE 399: zonesT:  "Speed zones (1→5)",
LINE 406: dir:     ["Through","Around","Over"],
LINE 407: typ:     ["Pass","Cross","Prog"],
LINE 404: crossesT:"Crosses",
LINE 405: total:   "Total",


## 23. Validate the Cross Data Structure

The source uses `cross[6]` as the total number of crosses.

Before creating the final columns, we will verify the structure
of the `cross` field across all 1,029 player records.

We will check:

- How many records contain a list
- The unique list lengths
- Minimum and maximum lengths
- A few example values

No data will be changed or saved yet.

In [27]:
cross_lengths = [
    len(player["cross"])
    for player in physical_players
    if isinstance(player.get("cross"), list)
]

print("CROSS FIELD STRUCTURE")
print("=" * 80)

print("Records with list:", len(cross_lengths))

if cross_lengths:

    print("Unique lengths:", sorted(set(cross_lengths)))
    print("Minimum length:", min(cross_lengths))
    print("Maximum length:", max(cross_lengths))

    print("\nFirst 10 examples:")
    print("-" * 80)

    shown = 0

    for player in physical_players:

        value = player.get("cross")

        if isinstance(value, list):

            print(
                f"{player['name']} ({player['team']}):",
                value
            )

            shown += 1

            if shown == 10:
                break

print("\nDone.")

CROSS FIELD STRUCTURE
Records with list: 1029
Unique lengths: [7]
Minimum length: 7
Maximum length: 7

First 10 examples:
--------------------------------------------------------------------------------
RODRI (Spain): [1, 0, 0, 2, 0, 0, 3]
Alexis MAC ALLISTER (Argentina): [0, 1, 0, 0, 0, 0, 1]
Marc CUCURELLA (Spain): [0, 1, 1, 2, 0, 11, 15]
Pau CUBARSI (Spain): [0, 1, 0, 0, 0, 0, 1]
Michael OLISE (France): [1, 2, 1, 0, 0, 0, 4]
Enzo FERNANDEZ (Argentina): [0, 3, 0, 0, 0, 2, 5]
Aymeric LAPORTE (Spain): [0, 0, 0, 0, 0, 0, 0]
Elliot ANDERSON (England): [1, 1, 0, 1, 0, 0, 3]
Harry KANE (England): [0, 1, 0, 0, 0, 2, 3]
Remo FREULER (Switzerland): [2, 1, 0, 0, 0, 1, 4]

Done.


## 24. Validate the Pass and Direction Data Structure

The source displays six `lbv` values:

Direction:
- Through
- Around
- Over

Type:
- Pass
- Cross
- Prog

The raw data appeared to contain seven values in some records.

We will inspect the complete `lbv` structure across all players
to determine the exact length and verify whether the seventh
value has a documented use.

No assumptions will be made about the seventh value.

In [28]:
lbv_lengths = [
    len(player["lbv"])
    for player in physical_players
    if isinstance(player.get("lbv"), list)
]

print("LBV FIELD STRUCTURE")
print("=" * 80)

print("Records with list:", len(lbv_lengths))

if lbv_lengths:

    print("Unique lengths:", sorted(set(lbv_lengths)))
    print("Minimum length:", min(lbv_lengths))
    print("Maximum length:", max(lbv_lengths))

    print("\nFirst 10 examples:")
    print("-" * 80)

    shown = 0

    for player in physical_players:

        value = player.get("lbv")

        if isinstance(value, list):

            print(
                f"{player['name']} ({player['team']}):",
                value
            )

            shown += 1

            if shown == 10:
                break

print("\nDone.")

LBV FIELD STRUCTURE
Records with list: 1029
Unique lengths: [6]
Minimum length: 6
Maximum length: 6

First 10 examples:
--------------------------------------------------------------------------------
RODRI (Spain): [78, 57, 52, 182, 0, 5]
Alexis MAC ALLISTER (Argentina): [31, 31, 14, 76, 0, 0]
Marc CUCURELLA (Spain): [20, 60, 29, 104, 1, 4]
Pau CUBARSI (Spain): [51, 117, 18, 177, 1, 8]
Michael OLISE (France): [52, 56, 25, 117, 3, 13]
Enzo FERNANDEZ (Argentina): [34, 69, 21, 119, 1, 4]
Aymeric LAPORTE (Spain): [42, 100, 40, 174, 0, 8]
Elliot ANDERSON (England): [44, 34, 34, 106, 2, 4]
Harry KANE (England): [13, 10, 13, 34, 0, 2]
Remo FREULER (Switzerland): [18, 38, 13, 66, 1, 2]

Done.


## 25. Validate Speed Zone Data

The source defines `zones` as:

`Speed zones (1→5)`

Every player record contains exactly five values.

We will validate that:

- Every player has five zone values.
- All values are numeric.
- No values are negative.
- The values can be used as distance measurements.

No data will be changed or saved yet.

In [29]:
zone_records = [
    player["zones"]
    for player in physical_players
    if isinstance(player.get("zones"), list)
]

print("SPEED ZONE VALIDATION")
print("=" * 80)

print("Records with zones:", len(zone_records))

zone_lengths = [len(z) for z in zone_records]

print("Unique lengths:", sorted(set(zone_lengths)))

numeric_values = all(
    all(isinstance(value, (int, float)) for value in zones)
    for zones in zone_records
)

non_negative = all(
    all(value >= 0 for value in zones)
    for zones in zone_records
)

print("All values numeric:", numeric_values)
print("All values non-negative:", non_negative)

if zone_records:

    flat_zones = [
        value
        for zones in zone_records
        for value in zones
    ]

    print("Minimum value:", min(flat_zones))
    print("Maximum value:", max(flat_zones))

print("\nDone.")

SPEED ZONE VALIDATION
Records with zones: 1029
Unique lengths: [5]
All values numeric: True
All values non-negative: True
Minimum value: 0
Maximum value: 44470.3

Done.


## 26. Identify the Cross Statistics Labels

The `cross` field contains seven values for every player.

The source code confirms that `cross[6]` represents the total
number of crosses.

Before creating the final dataset, we need to identify the
meaning of `cross[0]` through `cross[5]`.

We will inspect the `crossBlock()` function and the label
definitions used by it.

No assumptions will be made about the six individual values.

In [30]:
print("CROSS FUNCTION AND LABEL USAGE")
print("=" * 100)

for line_number, line in enumerate(
    physical_en_lines,
    start=1
):

    if 528 <= line_number <= 545:

        print(
            f"LINE {line_number}: "
            f"{line.strip()}"
        )

print("\nSearching for cross array indexes:")
print("=" * 100)

for line_number, line in enumerate(
    physical_en_lines,
    start=1
):

    if "cross[" in line.lower():

        print(
            f"LINE {line_number}: "
            f"{line.strip()}"
        )

print("\nDone.")

CROSS FUNCTION AND LABEL USAGE
LINE 528: + '  —  ' + LBL.typ[0] + ' ' + (+lbv[3]||0) + ' · ' + LBL.typ[1] + ' ' + (+lbv[4]||0) + ' · ' + LBL.typ[2] + ' ' + (+lbv[5]||0);
LINE 529: return '<div class="ph-block"><div class="ph-blk-t">' + LBL.lbT + '</div><div class="ph-chips">'
LINE 530: + chip(att, LBL.att) + chip(comp, LBL.comp) + chip(pct + '%', LBL.acc) + '</div><p class="ph-line">' + line + '</p></div>';
LINE 531: }
LINE 532: function crossBlock(cross){
LINE 533: var total = +cross[6] || 0; if (total <= 0) return '';
LINE 534: return '<div class="ph-block"><div class="ph-blk-t">' + LBL.crossesT + '</div><div class="ph-chips">' + chip(total, LBL.total) + '</div></div>';
LINE 535: }
LINE 536: function toggleDetail(tr){
LINE 537: if (tr.classList.contains('is-open')) { closeDetails(); return; }
LINE 538: closeDetails();
LINE 539: tr.classList.add('is-open');
LINE 540: var p = tr._p || {}, m = val(tr, 'm') || 1;
LINE 541: var f = [ (val(tr,'dist')/m)/(MAX.dist||1), (val(tr,'sprints')/m)

## 27. Validate Total Crosses

The source explicitly identifies `cross[6]` as the total number
of crosses.

The first six values are not individually labeled by the source
code we inspected.

We will test whether `cross[6]` equals the sum of
`cross[0]` through `cross[5]` for every player.

If this relationship holds, we can safely use the documented
Total Crosses value without assigning unsupported meanings to
the individual components.

In [31]:
cross_validation = []

for player in physical_players:

    cross = player.get("cross")

    if isinstance(cross, list) and len(cross) == 7:

        component_total = sum(
            float(value)
            for value in cross[:6]
        )

        documented_total = float(cross[6])

        cross_validation.append(
            component_total == documented_total
        )

print("TOTAL CROSS VALIDATION")
print("=" * 80)

print("Players checked:", len(cross_validation))
print(
    "Components sum to cross[6] for all players:",
    all(cross_validation)
)

print(
    "Mismatches:",
    sum(not result for result in cross_validation)
)

print("\nDone.")

TOTAL CROSS VALIDATION
Players checked: 1029
Components sum to cross[6] for all players: True
Mismatches: 0

Done.


## 28. Validate Main Physical Statistics

The source provides the following player-level physical fields:

- `m` — matches
- `dist` — distance
- `sprints` — sprints
- `hsr` — high-speed running
- `top` — top speed

We will verify that these fields exist for all player records,
contain numeric values, and do not contain negative values.

No data will be changed or saved yet.

In [32]:
scalar_fields = [
    "m",
    "dist",
    "sprints",
    "hsr",
    "top"
]

print("MAIN PHYSICAL STATISTICS VALIDATION")
print("=" * 80)

for field in scalar_fields:

    values = [
        player.get(field)
        for player in physical_players
    ]

    numeric = all(
        isinstance(value, (int, float))
        for value in values
    )

    non_negative = all(
        value >= 0
        for value in values
        if isinstance(value, (int, float))
    )

    missing = sum(
        value is None
        for value in values
    )

    print(f"\nField: {field}")
    print("Records:", len(values))
    print("Missing:", missing)
    print("All numeric:", numeric)
    print("All non-negative:", non_negative)

    if numeric and values:
        print("Minimum:", min(values))
        print("Maximum:", max(values))

print("\nDone.")

MAIN PHYSICAL STATISTICS VALIDATION

Field: m
Records: 1029
Missing: 0
All numeric: True
All non-negative: True
Minimum: 1
Maximum: 8

Field: dist
Records: 1029
Missing: 0
All numeric: True
All non-negative: True
Minimum: 90
Maximum: 96233.70000000001

Field: sprints
Records: 1029
Missing: 0
All numeric: True
All non-negative: True
Minimum: 0
Maximum: 403

Field: hsr
Records: 1029
Missing: 0
All numeric: True
All non-negative: True
Minimum: 1
Maximum: 1294

Field: top
Records: 1029
Missing: 0
All numeric: True
All non-negative: True
Minimum: 18.1
Maximum: 37.6

Done.


## 29. Validate Physical Players Against the Players Dataset

The physical endpoint contains 1,029 player records and provides:

- Player name
- Team
- Jersey number

Our existing `Players.csv` contains the official 1,248-player
FIFA squad list with:

- PlayerID
- PlayerName
- TeamID
- Position
- JerseyNumber

We will load `Players.csv` and test whether Team + JerseyNumber
provides a reliable way to connect the physical records to our
PlayerID values.

No physical data will be modified or saved yet.

In [33]:
data_folder = Path.cwd().parent / "Data"

players_df = pd.read_csv(
    data_folder / "Players.csv"
)

teams_df = pd.read_csv(
    data_folder / "Teams.csv"
)

print("Existing Players dataset:")
print("Shape:", players_df.shape)
print("Columns:", list(players_df.columns))

print("\nPhysical dataset:")
print("Records:", len(physical_players))

print("\nPhysical player sample:")
print(
    pd.DataFrame(physical_players)[
        ["name", "team", "num"]
    ].head(10).to_string(index=False)
)

print("\nPlayers dataset sample:")
print(
    players_df.head(10).to_string(index=False)
)

print("\nDone.")

Existing Players dataset:
Shape: (1248, 5)
Columns: ['PlayerID', 'PlayerName', 'TeamID', 'Position', 'JerseyNumber']

Physical dataset:
Records: 1029

Physical player sample:
               name        team  num
              RODRI       Spain   16
Alexis MAC ALLISTER   Argentina   20
     Marc CUCURELLA       Spain   24
        Pau CUBARSI       Spain   22
      Michael OLISE      France   11
     Enzo FERNANDEZ   Argentina   24
    Aymeric LAPORTE       Spain   14
    Elliot ANDERSON     England    8
         Harry KANE     England    9
       Remo FREULER Switzerland    8

Players dataset sample:
 PlayerID                    PlayerName  TeamID Position  JerseyNumber
        1          Melvin Feycal MASTIL      25       GK             1
        2                   Aissa MANDI      25       DF             2
        3                  Achref ABADA      25       DF             3
        4          Mohamed Amine TOUGAI      25       DF             4
        5              Zineddine BELAÏ

## 30. Match Physical Players to PlayerID

The physical data provides the player's team and jersey number.

The official Players dataset also contains the team and jersey
number, along with our internal PlayerID.

We will match the two datasets using:

    TeamID + JerseyNumber

This avoids relying on player-name spelling or formatting.

We will measure:

- Successfully matched physical players
- Unmatched physical players
- Duplicate matches

No data will be saved yet.

In [34]:
# Build team-name mapping from the existing Teams dataset
team_name_to_id = dict(
    zip(
        teams_df["TeamName"],
        teams_df["TeamID"]
    )
)

# Add TeamID to physical records
physical_match_df = pd.DataFrame(physical_players).copy()

physical_match_df["TeamID"] = (
    physical_match_df["team"]
    .map(team_name_to_id)
)

# Match using TeamID + JerseyNumber
players_key = players_df[
    ["PlayerID", "TeamID", "JerseyNumber"]
].copy()

physical_match_df = physical_match_df.merge(
    players_key,
    left_on=["TeamID", "num"],
    right_on=["TeamID", "JerseyNumber"],
    how="left"
)

matched = physical_match_df["PlayerID"].notna()

print("PLAYER MATCHING VALIDATION")
print("=" * 80)

print("Physical records:", len(physical_match_df))
print("Matched records:", matched.sum())
print("Unmatched records:", (~matched).sum())

print(
    "Duplicate PlayerID matches:",
    physical_match_df.loc[matched, "PlayerID"].duplicated().sum()
)

print(
    "Missing TeamID:",
    physical_match_df["TeamID"].isna().sum()
)

print("\nSample matched records:")
print(
    physical_match_df[
        ["name", "team", "num", "TeamID", "PlayerID"]
    ].head(15).to_string(index=False)
)

print("\nDone.")

PLAYER MATCHING VALIDATION
Physical records: 1029
Matched records: 863
Unmatched records: 166
Duplicate PlayerID matches: 0
Missing TeamID: 166

Sample matched records:
               name        team  num  TeamID  PlayerID
              RODRI       Spain   16    10.0    1056.0
Alexis MAC ALLISTER   Argentina   20     1.0      46.0
     Marc CUCURELLA       Spain   24    10.0    1064.0
        Pau CUBARSI       Spain   22    10.0    1062.0
      Michael OLISE      France   11     6.0     479.0
     Enzo FERNANDEZ   Argentina   24     1.0      50.0
    Aymeric LAPORTE       Spain   14    10.0    1054.0
    Elliot ANDERSON     England    8     5.0     450.0
         Harry KANE     England    9     5.0     451.0
       Remo FREULER Switzerland    8    11.0    1100.0
    Dayot UPAMECANO      France    4     6.0     472.0
    Jude BELLINGHAM     England   10     5.0     452.0
       Granit XHAKA Switzerland   10    11.0    1102.0
     Julian ALVAREZ   Argentina    9     1.0      35.0
      

## 30.1 Inspect Unmatched Team Names

The physical dataset contains 1,029 players, but 166 records
could not be assigned a TeamID.

The missing TeamIDs indicate that some team names in the
physical API use different spellings or naming conventions than
our official Teams dataset.

We will list the unmatched team names and their frequencies.

No values will be changed yet.

In [35]:
unmatched_teams = (
    physical_match_df.loc[
        physical_match_df["TeamID"].isna(),
        "team"
    ]
    .value_counts()
)

print("UNMATCHED TEAM NAMES")
print("=" * 80)

print(
    "Number of unique unmatched team names:",
    len(unmatched_teams)
)

print(
    "Number of unmatched physical records:",
    unmatched_teams.sum()
)

print("\nUnmatched teams:")
print("-" * 80)

for team, count in unmatched_teams.items():
    print(f"{team}: {count}")

print("\nDone.")

UNMATCHED TEAM NAMES
Number of unique unmatched team names: 8
Number of unmatched physical records: 166

Unmatched teams:
--------------------------------------------------------------------------------
Bosnia & Herzegovina: 23
Ivory Coast: 22
Cape Verde: 21
DR Congo: 21
Czech Republic: 21
Iran: 20
South Korea: 20
Turkey: 18

Done.


## 30.2 Apply Verified Team Name Aliases

The physical API uses eight team names that differ from the
names in our official Teams dataset.

These aliases were already verified during the MatchStatistics
collection notebook.

We will apply the same verified mappings:

- Bosnia & Herzegovina → Bosnia and Herzegovina
- Ivory Coast → Côte d'Ivoire
- Cape Verde → Cabo Verde
- DR Congo → Congo DR
- Czech Republic → Czechia
- Iran → IR Iran
- South Korea → Korea Republic
- Turkey → Türkiye

After applying the aliases, we will repeat the TeamID +
JerseyNumber matching.

No player statistics are being changed.

In [36]:
team_aliases = {
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "Cape Verde": "Cabo Verde",
    "Czech Republic": "Czechia",
    "DR Congo": "Congo DR",
    "Iran": "IR Iran",
    "Ivory Coast": "Côte d'Ivoire",
    "South Korea": "Korea Republic",
    "Turkey": "Türkiye"
}

# Apply the verified aliases
physical_match_df["team_normalized"] = (
    physical_match_df["team"]
    .replace(team_aliases)
)

# Map normalized team names to our TeamID
physical_match_df["TeamID"] = (
    physical_match_df["team_normalized"]
    .map(team_name_to_id)
)

# Match again using TeamID + JerseyNumber
physical_match_df = physical_match_df.drop(
    columns=["PlayerID", "JerseyNumber"],
    errors="ignore"
)

physical_match_df = physical_match_df.merge(
    players_key,
    left_on=["TeamID", "num"],
    right_on=["TeamID", "JerseyNumber"],
    how="left"
)

matched = physical_match_df["PlayerID"].notna()

print("PLAYER MATCHING AFTER ALIAS NORMALIZATION")
print("=" * 80)

print("Physical records:", len(physical_match_df))
print("Matched records:", matched.sum())
print("Unmatched records:", (~matched).sum())

print(
    "Missing TeamID:",
    physical_match_df["TeamID"].isna().sum()
)

print(
    "Duplicate PlayerID matches:",
    physical_match_df.loc[
        matched,
        "PlayerID"
    ].duplicated().sum()
)

print("\nUnmatched team names:")
print(
    physical_match_df.loc[
        physical_match_df["TeamID"].isna(),
        "team"
    ].value_counts()
)

print("\nSample matched records:")
print(
    physical_match_df[
        ["name", "team", "team_normalized", "num",
         "TeamID", "PlayerID"]
    ].head(15).to_string(index=False)
)

print("\nDone.")

PLAYER MATCHING AFTER ALIAS NORMALIZATION
Physical records: 1029
Matched records: 1029
Unmatched records: 0
Missing TeamID: 0
Duplicate PlayerID matches: 0

Unmatched team names:
Series([], Name: count, dtype: int64)

Sample matched records:
               name        team team_normalized  num  TeamID  PlayerID
              RODRI       Spain           Spain   16      10      1056
Alexis MAC ALLISTER   Argentina       Argentina   20       1        46
     Marc CUCURELLA       Spain           Spain   24      10      1064
        Pau CUBARSI       Spain           Spain   22      10      1062
      Michael OLISE      France          France   11       6       479
     Enzo FERNANDEZ   Argentina       Argentina   24       1        50
    Aymeric LAPORTE       Spain           Spain   14      10      1054
    Elliot ANDERSON     England         England    8       5       450
         Harry KANE     England         England    9       5       451
       Remo FREULER Switzerland     Switzerland 

## 31. Validate Physical Player Coverage

The official FIFA squad dataset contains 1,248 players, while
the physical endpoint contains 1,029 players.

We will compare the two datasets using PlayerID and determine:

- Number of unique physical PlayerIDs
- Number of squad players with physical data
- Number of squad players without physical data
- Whether any PlayerID appears more than once

The 1,029 physical records will remain unchanged.

In [37]:
physical_player_ids = physical_match_df["PlayerID"]

unique_physical_ids = physical_player_ids.nunique()

players_with_physical = players_df[
    players_df["PlayerID"].isin(physical_player_ids)
]

players_without_physical = players_df[
    ~players_df["PlayerID"].isin(physical_player_ids)
]

duplicate_player_ids = (
    physical_player_ids
    .duplicated()
    .sum()
)

print("PHYSICAL PLAYER COVERAGE")
print("=" * 80)

print("Official squad players:", len(players_df))
print("Physical player records:", len(physical_match_df))
print("Unique physical PlayerIDs:", unique_physical_ids)
print("Squad players with physical data:", len(players_with_physical))
print("Squad players without physical data:", len(players_without_physical))
print("Duplicate physical PlayerIDs:", duplicate_player_ids)

print("\nCoverage percentage:")
print(
    round(
        len(players_with_physical) / len(players_df) * 100,
        2
    ),
    "%"
)

print("\nDone.")

PHYSICAL PLAYER COVERAGE
Official squad players: 1248
Physical player records: 1029
Unique physical PlayerIDs: 1029
Squad players with physical data: 1029
Squad players without physical data: 219
Duplicate physical PlayerIDs: 0

Coverage percentage:
82.45 %

Done.


## 32. Inspect the Complete Physical Player Record

The physical dataset has been successfully matched to our
official PlayerID values.

Before creating the final dataset, we will inspect one complete
record after matching.

This allows us to verify all available source fields before
selecting the final SQL columns.

No data will be modified or exported yet.

In [38]:
print("COMPLETE MATCHED PHYSICAL RECORD")
print("=" * 100)

first_record = physical_match_df.iloc[0]

for column, value in first_record.items():

    print(f"{column}:")
    print(f"  {value}")
    print("-" * 80)

print("\nTotal columns:", len(physical_match_df.columns))
print("Columns:")
for column in physical_match_df.columns:
    print("-", column)

print("\nDone.")

COMPLETE MATCHED PHYSICAL RECORD
name:
  RODRI
--------------------------------------------------------------------------------
team:
  Spain
--------------------------------------------------------------------------------
teamAr:
  إسبانيا
--------------------------------------------------------------------------------
flag:
  https://foot-boll.com/flag.php?s=w40&c=es
--------------------------------------------------------------------------------
photo:
  https://digitalhub.fifa.com/transform/295ab5b2-4dbf-42b9-918b-04fb4ae0ec0a/RODRI_411375
--------------------------------------------------------------------------------
num:
  16
--------------------------------------------------------------------------------
m:
  8
--------------------------------------------------------------------------------
dist:
  96233.70000000001
--------------------------------------------------------------------------------
sprints:
  318
--------------------------------------------------------------------

## 33. Check Match Details for Player-Level Data

The `physical` endpoint provides tournament-level player data.

Our SQL design originally included a `PlayerMatchStats` table,
which requires statistics for individual players in individual
matches.

Before deciding how Notebook 06 will map to the database, we will
inspect a detailed match response for any player-level fields.

We will search the complete match response recursively for
player-related structures.

No data will be modified or saved.

In [39]:
def find_player_related_fields(obj, path="root"):
    results = []

    if isinstance(obj, dict):

        for key, value in obj.items():

            key_lower = str(key).lower()

            if any(
                term in key_lower
                for term in [
                    "player",
                    "lineup",
                    "substitution",
                    "appearance"
                ]
            ):
                results.append(
                    (f"{path}.{key}", value)
                )

            results.extend(
                find_player_related_fields(
                    value,
                    f"{path}.{key}"
                )
            )

    elif isinstance(obj, list):

        for index, item in enumerate(obj):

            results.extend(
                find_player_related_fields(
                    item,
                    f"{path}[{index}]"
                )
            )

    return results


player_fields = find_player_related_fields(
    detailed_match
)

print("PLAYER-RELATED FIELDS IN MATCH RESPONSE")
print("=" * 100)

print("Number of player-related fields found:", len(player_fields))

for path, value in player_fields:

    print("\nPATH:", path)
    print("-" * 100)

    if isinstance(value, (dict, list)):

        print(
            "Type:",
            type(value).__name__
        )

        if isinstance(value, list):
            print("Length:", len(value))

            if value:
                print("First item:", value[0])

        else:
            print("Keys:", list(value.keys()))

    else:
        print("Value:", value)

print("\nDone.")

PLAYER-RELATED FIELDS IN MATCH RESPONSE
Number of player-related fields found: 0

Done.


## 34. Verify Physical Data Units

Before creating the final SQL-ready dataset, we need to verify
the units used by the source for the physical statistics.

The physical page transforms the raw API values before displaying
them.

We will inspect the English page code for the units and display
labels associated with:

- `dist`
- `sprints`
- `hsr`
- `top`

This will allow us to store the values with clear and correct
column names.

In [40]:
unit_terms = [
    "LBL.km",
    "LBL.kmh",
    "LBL.perM",
    "LBL.dist",
    "LBL.sprints",
    "LBL.hsr",
    "LBL.top"
]

print("PHYSICAL STATISTICS UNIT MAPPINGS")
print("=" * 100)

for line_number, line in enumerate(
    physical_en_lines,
    start=1
):

    if any(
        term.lower() in line.lower()
        for term in unit_terms
    ):

        print(
            f"LINE {line_number}: "
            f"{line.strip()}"
        )

print("\nDone.")

PHYSICAL STATISTICS UNIT MAPPINGS
LINE 506: var labels = [LBL.dist, LBL.sprints, LBL.hsr, LBL.top], anc = ['middle','start','middle','end'], dy = [-10, 0, 22, 0], labs = '';
LINE 542: var vals = [ (val(tr,'dist')/1000/m).toFixed(1) + ' ' + LBL.km, (val(tr,'sprints')/m).toFixed(1), (val(tr,'hsr')/m).toFixed(1), (Math.round(val(tr,'top')*10)/10) + ' ' + LBL.kmh ];
LINE 543: var chips = chip((val(tr,'dist')/1000/m).toFixed(1), LBL.dist + ' ' + LBL.km + ' /' + LBL.perM)
LINE 544: + chip((val(tr,'sprints')/m).toFixed(1), LBL.sprints + ' /' + LBL.perM)
LINE 545: + chip((val(tr,'hsr')/m).toFixed(1), LBL.hsr + ' /' + LBL.perM)
LINE 546: + chip((Math.round(val(tr,'top')*10)/10) + ' ' + LBL.kmh, LBL.top);

Done.


## 35. Validate the `lb` Field Structure

The physical data contains both:

- `lb`
- `lbv`

The source uses them together in the `lbBlock()` function.

We have already established the six meanings of `lbv`, but
we have not yet validated the structure of `lb`.

We will check:

- Number of records containing `lb`
- List length
- Numeric values
- Non-negative values
- Sample records

No data will be modified or saved.

In [41]:
lb_lengths = [
    len(player["lb"])
    for player in physical_players
    if isinstance(player.get("lb"), list)
]

print("LB FIELD STRUCTURE")
print("=" * 80)

print("Records with list:", len(lb_lengths))

if lb_lengths:

    print("Unique lengths:", sorted(set(lb_lengths)))
    print("Minimum length:", min(lb_lengths))
    print("Maximum length:", max(lb_lengths))

    lb_records = [
        player["lb"]
        for player in physical_players
        if isinstance(player.get("lb"), list)
    ]

    numeric = all(
        all(
            isinstance(value, (int, float))
            for value in lb
        )
        for lb in lb_records
    )

    non_negative = all(
        all(
            value >= 0
            for value in lb
        )
        for lb in lb_records
    )

    print("All values numeric:", numeric)
    print("All values non-negative:", non_negative)

    print("\nFirst 10 examples:")
    print("-" * 80)

    shown = 0

    for player in physical_players:

        value = player.get("lb")

        if isinstance(value, list):

            print(
                f"{player['name']} ({player['team']}):",
                value
            )

            shown += 1

            if shown == 10:
                break

print("\nDone.")

LB FIELD STRUCTURE
Records with list: 1029
Unique lengths: [2]
Minimum length: 2
Maximum length: 2
All values numeric: True
All values non-negative: True

First 10 examples:
--------------------------------------------------------------------------------
RODRI (Spain): [187, 171]
Alexis MAC ALLISTER (Argentina): [76, 63]
Marc CUCURELLA (Spain): [109, 87]
Pau CUBARSI (Spain): [186, 178]
Michael OLISE (France): [133, 103]
Enzo FERNANDEZ (Argentina): [124, 111]
Aymeric LAPORTE (Spain): [182, 160]
Elliot ANDERSON (England): [112, 89]
Harry KANE (England): [36, 24]
Remo FREULER (Switzerland): [69, 59]

Done.


## 36. Identify the `lb` Statistics

The source uses the `lb` field together with `lbv` inside
`lbBlock()`.

We have validated that `lb` contains exactly two values for
every player.

Before including these values in the final dataset, we will
inspect the source code that calculates and displays `lb` to
identify the exact labels for the two values.

No assumptions will be made.

In [42]:
print("LB FIELD LABEL USAGE")
print("=" * 100)

for line_number, line in enumerate(
    physical_en_lines,
    start=1
):

    if (
        "lbBlock" in line
        or "var att" in line
        or "var comp" in line
        or "LBL.att" in line
        or "LBL.comp" in line
    ):

        print(
            f"LINE {line_number}: "
            f"{line.strip()}"
        )

print("\nSearching for label definitions:")
print("=" * 100)

lb_label_patterns = [
    r"lbT\s*:",
    r"att\s*:",
    r"comp\s*:",
    r"acc\s*:"
]

for pattern in lb_label_patterns:

    for line_number, line in enumerate(
        physical_en_lines,
        start=1
    ):

        if re.search(pattern, line, re.IGNORECASE):

            print(
                f"LINE {line_number}: "
                f"{line.strip()}"
            )

print("\nDone.")

LB FIELD LABEL USAGE
LINE 524: function lbBlock(lb, lbv){
LINE 525: var att = +lb[0] || 0; if (att <= 0) return '';
LINE 526: var comp = +lb[1] || 0, pct = Math.round(comp / att * 100);
LINE 530: + chip(att, LBL.att) + chip(comp, LBL.comp) + chip(pct + '%', LBL.acc) + '</div><p class="ph-line">' + line + '</p></div>';
LINE 547: var rich = bar(p.zones || []) + lbBlock(p.lb || [0, 0], p.lbv || []) + crossBlock(p.cross || []);

Searching for label definitions:
LINE 400: lbT:     "Line breaks",
LINE 401: att:     "Att",
LINE 402: comp:    "Comp",
LINE 403: acc:     "Acc",

Done.


## 37. Define the Final Player Physical Statistics Columns

The physical API provides validated player-level tournament data.

The final SQL-ready dataset will contain only fields whose
meaning has been verified from the source.

Core physical statistics:

- Matches
- Distance
- Sprints
- High-speed running
- Top speed

Speed zones:

- Speed Zone 1
- Speed Zone 2
- Speed Zone 3
- Speed Zone 4
- Speed Zone 5

Line breaks:

- Line Breaks Attempted
- Line Breaks Completed
- Line Breaks Accuracy Percentage

Passing/progression:

- Through
- Around
- Over
- Pass
- Cross
- Prog

Crosses:

- Total Crosses

The undocumented individual components of `cross[0]` through
`cross[5]` will NOT be included.

The dataset will be linked to the existing Players table using
PlayerID.

No export will be performed in this step.

In [43]:
final_columns = [
    "PlayerID",
    "Matches",
    "DistanceMeters",
    "Sprints",
    "HighSpeedRunning",
    "TopSpeedKmh",
    "SpeedZone1",
    "SpeedZone2",
    "SpeedZone3",
    "SpeedZone4",
    "SpeedZone5",
    "LineBreaksAttempted",
    "LineBreaksCompleted",
    "LineBreaksAccuracyPct",
    "Through",
    "Around",
    "Over",
    "Pass",
    "Cross",
    "Prog",
    "TotalCrosses"
]

print("FINAL PLAYER PHYSICAL DATASET COLUMNS")
print("=" * 80)

print("Number of columns:", len(final_columns))

for i, column in enumerate(final_columns, start=1):
    print(f"{i:2}. {column}")

print("\nDone.")

FINAL PLAYER PHYSICAL DATASET COLUMNS
Number of columns: 21
 1. PlayerID
 2. Matches
 3. DistanceMeters
 4. Sprints
 5. HighSpeedRunning
 6. TopSpeedKmh
 7. SpeedZone1
 8. SpeedZone2
 9. SpeedZone3
10. SpeedZone4
11. SpeedZone5
12. LineBreaksAttempted
13. LineBreaksCompleted
14. LineBreaksAccuracyPct
15. Through
16. Around
17. Over
18. Pass
19. Cross
20. Prog
21. TotalCrosses

Done.


## 38. Build the Final Player Physical Dataset

We will now transform the validated physical API records into
the final SQL-ready structure.

The dataset will:

- Use our existing `PlayerID`
- Keep the raw physical measurements
- Flatten the validated array fields
- Calculate LineBreaksAccuracyPct from Attempted and Completed
- Use Total Crosses from `cross[6]`
- Contain exactly 1,029 player records

No CSV will be exported yet.

In [44]:
physical_final_df = pd.DataFrame({
    "PlayerID": physical_match_df["PlayerID"].astype(int),

    "Matches": physical_match_df["m"].astype(int),

    "DistanceMeters": physical_match_df["dist"].astype(float),

    "Sprints": physical_match_df["sprints"].astype(int),

    "HighSpeedRunning": physical_match_df["hsr"].astype(float),

    "TopSpeedKmh": physical_match_df["top"].astype(float),

    "SpeedZone1": physical_match_df["zones"].apply(lambda x: float(x[0])),
    "SpeedZone2": physical_match_df["zones"].apply(lambda x: float(x[1])),
    "SpeedZone3": physical_match_df["zones"].apply(lambda x: float(x[2])),
    "SpeedZone4": physical_match_df["zones"].apply(lambda x: float(x[3])),
    "SpeedZone5": physical_match_df["zones"].apply(lambda x: float(x[4])),

    "LineBreaksAttempted": physical_match_df["lb"].apply(lambda x: int(x[0])),
    "LineBreaksCompleted": physical_match_df["lb"].apply(lambda x: int(x[1])),

    "LineBreaksAccuracyPct": physical_match_df["lb"].apply(
        lambda x: round((x[1] / x[0]) * 100, 2)
        if x[0] > 0 else 0
    ),

    "Through": physical_match_df["lbv"].apply(lambda x: int(x[0])),
    "Around": physical_match_df["lbv"].apply(lambda x: int(x[1])),
    "Over": physical_match_df["lbv"].apply(lambda x: int(x[2])),
    "Pass": physical_match_df["lbv"].apply(lambda x: int(x[3])),
    "Cross": physical_match_df["lbv"].apply(lambda x: int(x[4])),
    "Prog": physical_match_df["lbv"].apply(lambda x: int(x[5])),

    "TotalCrosses": physical_match_df["cross"].apply(
        lambda x: int(x[6])
    )
})

print("FINAL DATASET CREATED")
print("=" * 80)

print("Shape:", physical_final_df.shape)

print("\nColumns:")
for column in physical_final_df.columns:
    print("-", column)

print("\nFirst 10 rows:")
print(
    physical_final_df.head(10).to_string(index=False)
)

print("\nDone.")

FINAL DATASET CREATED
Shape: (1029, 21)

Columns:
- PlayerID
- Matches
- DistanceMeters
- Sprints
- HighSpeedRunning
- TopSpeedKmh
- SpeedZone1
- SpeedZone2
- SpeedZone3
- SpeedZone4
- SpeedZone5
- LineBreaksAttempted
- LineBreaksCompleted
- LineBreaksAccuracyPct
- Through
- Around
- Over
- Pass
- Cross
- Prog
- TotalCrosses

First 10 rows:
 PlayerID  Matches  DistanceMeters  Sprints  HighSpeedRunning  TopSpeedKmh  SpeedZone1  SpeedZone2  SpeedZone3  SpeedZone4  SpeedZone5  LineBreaksAttempted  LineBreaksCompleted  LineBreaksAccuracyPct  Through  Around  Over  Pass  Cross  Prog  TotalCrosses
     1056        8         96233.7      318            1294.0         31.5     30824.8     44470.3     15304.9      4504.4      1129.3                  187                  171                  91.44       78      57    52   182      0     5             3
       46        8         90561.9      277            1024.0         30.2     32710.0     41710.2     11818.2      3470.1       853.3           

## 39. Validate the Final Player Physical Dataset

The final dataset contains 1,029 player records and 21 columns.

Before exporting it, we will validate:

- Row count
- Column structure
- Duplicate PlayerIDs
- Missing values
- Invalid PlayerIDs
- Negative statistics
- Line-break accuracy range
- Completed line breaks greater than attempts
- Total crosses consistency
- Matches greater than zero

The dataset will not be exported until all validations pass.

In [45]:
print("FINAL PLAYER PHYSICAL DATASET VALIDATION")
print("=" * 100)

# Basic structure
print("Rows:", len(physical_final_df))
print("Columns:", len(physical_final_df.columns))
print(
    "Column structure correct:",
    list(physical_final_df.columns) == final_columns
)

# PlayerID validation
print(
    "Duplicate PlayerIDs:",
    physical_final_df["PlayerID"].duplicated().sum()
)

print(
    "Missing PlayerIDs:",
    physical_final_df["PlayerID"].isna().sum()
)

print(
    "Invalid PlayerIDs:",
    (~physical_final_df["PlayerID"].isin(players_df["PlayerID"])).sum()
)

# Missing values
print(
    "Total missing values:",
    physical_final_df.isna().sum().sum()
)

# Numeric validation
numeric_columns = [
    column
    for column in final_columns
    if column != "PlayerID"
]

negative_values = (
    physical_final_df[numeric_columns] < 0
).sum().sum()

print(
    "Negative values:",
    negative_values
)

# Matches
print(
    "Players with Matches <= 0:",
    (physical_final_df["Matches"] <= 0).sum()
)

# Line breaks
print(
    "Completed > Attempted:",
    (
        physical_final_df["LineBreaksCompleted"]
        > physical_final_df["LineBreaksAttempted"]
    ).sum()
)

print(
    "Accuracy outside 0-100:",
    (
        (physical_final_df["LineBreaksAccuracyPct"] < 0)
        |
        (physical_final_df["LineBreaksAccuracyPct"] > 100)
    ).sum()
)

# Cross validation
cross_sum = (
    physical_match_df["cross"]
    .apply(
        lambda x: sum(float(value) for value in x[:6])
        == float(x[6])
    )
)

print(
    "Total Crosses consistency failures:",
    (~cross_sum).sum()
)

print("\nDone.")

FINAL PLAYER PHYSICAL DATASET VALIDATION
Rows: 1029
Columns: 21
Column structure correct: True
Duplicate PlayerIDs: 0
Missing PlayerIDs: 0
Invalid PlayerIDs: 0
Total missing values: 0
Negative values: 0
Players with Matches <= 0: 0
Completed > Attempted: 0
Accuracy outside 0-100: 0
Total Crosses consistency failures: 0

Done.


## 40. Export Player Physical Statistics

The final player physical dataset has passed all validation checks.

We will now export the dataset as:

`PlayerPhysicalStats.csv`

The file will contain the 1,029 real player records and the
21 validated columns.

After exporting, we will reload the CSV to verify that the
saved file is identical in structure and row count.

In [46]:
output_path = data_folder / "PlayerPhysicalStats.csv"

physical_final_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("EXPORT SUCCESSFUL")
print("=" * 80)

print("File:", output_path)
print("Rows:", len(physical_final_df))
print("Columns:", len(physical_final_df.columns))
print("File exists:", output_path.exists())

if output_path.exists():
    print("File size:", output_path.stat().st_size, "bytes")

print("\nDone.")

EXPORT SUCCESSFUL
File: d:\ai course hw\FIFA_WorldCup_2026_Project\Data\PlayerPhysicalStats.csv
Rows: 1029
Columns: 21
File exists: True
File size: 106190 bytes

Done.


## 40.1 Reload and Verify the Exported CSV

The PlayerPhysicalStats dataset has been exported successfully.

We will reload the saved CSV and verify that:

- It contains 1,029 rows.
- It contains 21 columns.
- The column structure is unchanged.
- There are no missing values.
- PlayerID remains unique.
- All PlayerIDs remain valid foreign keys to Players.
- The saved data matches the in-memory dataset structure.

This confirms that the exported file is ready for the SQL stage.

In [47]:
physical_check_df = pd.read_csv(
    output_path
)

print("EXPORTED CSV VERIFICATION")
print("=" * 100)

print("Shape:", physical_check_df.shape)

print(
    "Correct row count:",
    len(physical_check_df) == 1029
)

print(
    "Correct column count:",
    len(physical_check_df.columns) == 21
)

print(
    "Correct column structure:",
    list(physical_check_df.columns) == final_columns
)

print(
    "Missing values:",
    physical_check_df.isna().sum().sum()
)

print(
    "Duplicate PlayerIDs:",
    physical_check_df["PlayerID"].duplicated().sum()
)

print(
    "Invalid PlayerIDs:",
    (
        ~physical_check_df["PlayerID"]
        .isin(players_df["PlayerID"])
    ).sum()
)

print("\nPlayerID range:")
print(
    physical_check_df["PlayerID"].min(),
    "to",
    physical_check_df["PlayerID"].max()
)

print("\nFirst 5 rows:")
print(
    physical_check_df.head().to_string(index=False)
)

print("\nCSV verification complete.")

EXPORTED CSV VERIFICATION
Shape: (1029, 21)
Correct row count: True
Correct column count: True
Correct column structure: True
Missing values: 0
Duplicate PlayerIDs: 0
Invalid PlayerIDs: 0

PlayerID range:
2 to 1248

First 5 rows:
 PlayerID  Matches  DistanceMeters  Sprints  HighSpeedRunning  TopSpeedKmh  SpeedZone1  SpeedZone2  SpeedZone3  SpeedZone4  SpeedZone5  LineBreaksAttempted  LineBreaksCompleted  LineBreaksAccuracyPct  Through  Around  Over  Pass  Cross  Prog  TotalCrosses
     1056        8         96233.7      318            1294.0         31.5     30824.8     44470.3     15304.9      4504.4      1129.3                  187                  171                  91.44       78      57    52   182      0     5             3
       46        8         90561.9      277            1024.0         30.2     32710.0     41710.2     11818.2      3470.1       853.3                   76                   63                  82.89       31      31    14    76      0     0             1
  

## 45. Rename Cross Column for SQL Compatibility

The SQL table uses the column name `Crosses`.

We will rename the corresponding CSV column from `Cross`
to `Crosses` so that the Python dataset and SQL database
use the same column structure.

In [48]:
physical_final_df = physical_final_df.rename(
    columns={"Cross": "Crosses"}
)

print("Column renamed successfully.")
print("Crosses" in physical_final_df.columns)
print("Cross" in physical_final_df.columns)

print("\nColumns:")
print(physical_final_df.columns.tolist())

Column renamed successfully.
True
False

Columns:
['PlayerID', 'Matches', 'DistanceMeters', 'Sprints', 'HighSpeedRunning', 'TopSpeedKmh', 'SpeedZone1', 'SpeedZone2', 'SpeedZone3', 'SpeedZone4', 'SpeedZone5', 'LineBreaksAttempted', 'LineBreaksCompleted', 'LineBreaksAccuracyPct', 'Through', 'Around', 'Over', 'Pass', 'Crosses', 'Prog', 'TotalCrosses']


## 46.0 Update Expected Column Structure

The CSV column was renamed from `Cross` to `Crosses`.

We will update the expected column list used by our
validation code so that it matches the final SQL schema
and exported CSV.

In [51]:
final_columns = [
    "PlayerID",
    "Matches",
    "DistanceMeters",
    "Sprints",
    "HighSpeedRunning",
    "TopSpeedKmh",
    "SpeedZone1",
    "SpeedZone2",
    "SpeedZone3",
    "SpeedZone4",
    "SpeedZone5",
    "LineBreaksAttempted",
    "LineBreaksCompleted",
    "LineBreaksAccuracyPct",
    "Through",
    "Around",
    "Over",
    "Pass",
    "Crosses",
    "Prog",
    "TotalCrosses"
]

print("Expected column list updated.")
print("Number of expected columns:", len(final_columns))
print("\nColumns:")
print(final_columns)

Expected column list updated.
Number of expected columns: 21

Columns:
['PlayerID', 'Matches', 'DistanceMeters', 'Sprints', 'HighSpeedRunning', 'TopSpeedKmh', 'SpeedZone1', 'SpeedZone2', 'SpeedZone3', 'SpeedZone4', 'SpeedZone5', 'LineBreaksAttempted', 'LineBreaksCompleted', 'LineBreaksAccuracyPct', 'Through', 'Around', 'Over', 'Pass', 'Crosses', 'Prog', 'TotalCrosses']


In [52]:
print("FINAL CSV VERIFICATION")
print("=" * 100)

print("Shape:", physical_sql_check.shape)

print(
    "Correct columns:",
    list(physical_sql_check.columns) == final_columns
)

print(
    "Contains Crosses:",
    "Crosses" in physical_sql_check.columns
)

print(
    "Contains Cross:",
    "Cross" in physical_sql_check.columns
)

print(
    "Missing values:",
    physical_sql_check.isna().sum().sum()
)

print(
    "Duplicate PlayerIDs:",
    physical_sql_check["PlayerID"].duplicated().sum()
)

print(
    "Invalid PlayerIDs:",
    (~physical_sql_check["PlayerID"].isin(players_df["PlayerID"])).sum()
)

print("\nFinal CSV verification complete.")

FINAL CSV VERIFICATION
Shape: (1029, 21)
Correct columns: True
Contains Crosses: True
Contains Cross: False
Missing values: 0
Duplicate PlayerIDs: 0
Invalid PlayerIDs: 0

Final CSV verification complete.


In [53]:
output_path = data_folder / "PlayerPhysicalStats.csv"

physical_final_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("RE-EXPORT SUCCESSFUL")
print("=" * 80)
print("File:", output_path)
print("Rows:", len(physical_final_df))
print("Columns:", len(physical_final_df.columns))
print("File exists:", output_path.exists())
print("Contains Crosses:", "Crosses" in physical_final_df.columns)
print("Contains Cross:", "Cross" in physical_final_df.columns)

print("\nDone.")

RE-EXPORT SUCCESSFUL
File: d:\ai course hw\FIFA_WorldCup_2026_Project\Data\PlayerPhysicalStats.csv
Rows: 1029
Columns: 21
File exists: True
Contains Crosses: True
Contains Cross: False

Done.


## 27.1 Verify PlayerPhysicalStats CSV

Reload the PlayerPhysicalStats CSV and verify its structure,
primary keys, values, and relationship with the Players table.

In [1]:
import pandas as pd
from pathlib import Path

data_folder = Path.cwd().parent / "Data"

physical_path = data_folder / "PlayerPhysicalStats.csv"
players_path = data_folder / "Players.csv"

# Load both datasets
physical_df = pd.read_csv(physical_path)
players_df = pd.read_csv(players_path)

print("PLAYER PHYSICAL STATS VERIFICATION")
print("=" * 100)

print("Shape:", physical_df.shape)

print("\nColumns:")
print(physical_df.columns.tolist())

print("\nMissing values:")
print(physical_df.isna().sum().to_string())

print("\nDuplicate PlayerIDs:",
      physical_df["PlayerID"].duplicated().sum())

print(
    "\nPlayerID range:",
    physical_df["PlayerID"].min(),
    "to",
    physical_df["PlayerID"].max()
)

print("\nInvalid PlayerIDs:",
      (~physical_df["PlayerID"].isin(players_df["PlayerID"])).sum())

print("\nNegative values:",
      (physical_df.select_dtypes(include="number") < 0).sum().sum())

print("\nMatches <= 0:",
      (physical_df["Matches"] <= 0).sum())

print("\nCompleted > Attempted:",
      (physical_df["LineBreaksCompleted"] >
       physical_df["LineBreaksAttempted"]).sum())

print("\nAccuracy outside 0-100:",
      ((physical_df["LineBreaksAccuracyPct"] < 0) |
       (physical_df["LineBreaksAccuracyPct"] > 100)).sum())

print("\nTotal crosses consistency failures:",
      (
          physical_df[
              ["Pass", "Crosses", "Prog"]
          ].sum(axis=1) != physical_df["TotalCrosses"]
      ).sum())

PLAYER PHYSICAL STATS VERIFICATION
Shape: (1029, 21)

Columns:
['PlayerID', 'Matches', 'DistanceMeters', 'Sprints', 'HighSpeedRunning', 'TopSpeedKmh', 'SpeedZone1', 'SpeedZone2', 'SpeedZone3', 'SpeedZone4', 'SpeedZone5', 'LineBreaksAttempted', 'LineBreaksCompleted', 'LineBreaksAccuracyPct', 'Through', 'Around', 'Over', 'Pass', 'Crosses', 'Prog', 'TotalCrosses']

Missing values:
PlayerID                 0
Matches                  0
DistanceMeters           0
Sprints                  0
HighSpeedRunning         0
TopSpeedKmh              0
SpeedZone1               0
SpeedZone2               0
SpeedZone3               0
SpeedZone4               0
SpeedZone5               0
LineBreaksAttempted      0
LineBreaksCompleted      0
LineBreaksAccuracyPct    0
Through                  0
Around                   0
Over                     0
Pass                     0
Crosses                  0
Prog                     0
TotalCrosses             0

Duplicate PlayerIDs: 0

PlayerID range: 2 to 1248



## 27.1 Correct PlayerPhysicalStats Validation

Validate the PlayerPhysicalStats dataset using the actual structure
of the source API. TotalCrosses is an independent field and should
not be compared with Pass, Crosses, and Prog.

In [2]:
print("PLAYER PHYSICAL STATS CORRECTED VERIFICATION")
print("=" * 100)

print("Shape:", physical_df.shape)

print("\nDuplicate PlayerIDs:",
      physical_df["PlayerID"].duplicated().sum())

print(
    "PlayerID range:",
    physical_df["PlayerID"].min(),
    "to",
    physical_df["PlayerID"].max()
)

print("Invalid PlayerIDs:",
      (~physical_df["PlayerID"].isin(players_df["PlayerID"])).sum())

print("Missing values:",
      physical_df.isna().sum().sum())

print("Negative values:",
      (physical_df.select_dtypes(include="number") < 0).sum().sum())

print("Matches <= 0:",
      (physical_df["Matches"] <= 0).sum())

print("Completed > Attempted:",
      (
          physical_df["LineBreaksCompleted"]
          > physical_df["LineBreaksAttempted"]
      ).sum())

print("Accuracy outside 0-100:",
      (
          (physical_df["LineBreaksAccuracyPct"] < 0) |
          (physical_df["LineBreaksAccuracyPct"] > 100)
      ).sum())

print("\nFinal checks:")
print("1029 rows:", len(physical_df) == 1029)
print("21 columns:", len(physical_df.columns) == 21)
print("No duplicate PlayerIDs:",
      physical_df["PlayerID"].is_unique)
print("No missing values:",
      physical_df.isna().sum().sum() == 0)

PLAYER PHYSICAL STATS CORRECTED VERIFICATION
Shape: (1029, 21)

Duplicate PlayerIDs: 0
PlayerID range: 2 to 1248
Invalid PlayerIDs: 0
Missing values: 0
Negative values: 0
Matches <= 0: 0
Completed > Attempted: 0
Accuracy outside 0-100: 0

Final checks:
1029 rows: True
21 columns: True
No duplicate PlayerIDs: True
No missing values: True
